In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy.stats as stats

In [ ]:
# Read file
df = pd.read_excel('Assignment1Data.xlsx', sheet_name = 'StocksBondsBills')

# Create datetime and set as index
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY = 1))
df.set_index('Date', inplace=True)

# Save file for results
output_file = 'Assignment1_results.xlsx'

# Question 1

In [ ]:
# 1a)
# Excess returns and length
df['Excess_stocks'] = df['Stocks'] - df['Bills']
returns = df['Excess_stocks']
T = len(returns)

# Sample mean and variance
mu_hat = np.mean(returns)
sigma2_hat = np.mean((returns - mu_hat) ** 2)

# First and second sample moments
M1 = np.mean(returns - mu_hat)
M2 = np.mean((returns - mu_hat) ** 2 - sigma2_hat)

# g_T matrix
g_T = np.array([M1, M2])

# The D_T matrix becomes [-1 0] and eventually cancels out eventually leaving us only S_T necessary for computation
#                        [0 -1]
# S_T covariance matrix of moment conditions
# f_t matrix
f_t = np.column_stack([
    returns - mu_hat, 
    (returns - mu_hat) ** 2 - sigma2_hat
])

S_T = (f_t.T @ f_t) / T

# Since the system is exactly identified (q = k = 2) the weight matrix W_T is the identity matrix and does not impact the variance-covariance matrix
# Simiarily, the D_T' @ D_T also becomes the identity matrix we're left with V_theta_hat = S_T

V_theta_hat = S_T / T

# Standard Errors
se_mu_hat = np.sqrt(V_theta_hat[0, 0])
se_sigma2_hat = np.sqrt(V_theta_hat[1, 1])

# Output results
print(f"Estimated μ: {mu_hat:.6f}, Standard Error: {se_mu_hat:.6f}")
print(f"Estimated σ²: {sigma2_hat:.6f}, Standard Error: {se_sigma2_hat:.6f}")
print(f"Var-Covar Matrix - 0 0: {V_theta_hat[0, 0]} and 0 1: {V_theta_hat[0, 1]} and 1 0: {V_theta_hat[1, 0]} and 1 1: {V_theta_hat[1, 1]} ")


# Save to file
results = pd.DataFrame({
    'Statistic': ['Mu estimate', 'Variance estimate', 'Standard Error of Mu', 'Standard Error of Variance'],
    'Value': [mu_hat, sigma2_hat, se_mu_hat, se_sigma2_hat]
})

# Save to file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results.to_excel(writer, sheet_name='1a)mu_sigma_SE', index=False)

In [ ]:
# 1b)

# Deriving the covariance matrix results in matrix:
# [sigma2 / T    0]
# [0   2*sigma2 / T]

# Variance of mean and sigma2 under normality assumptions
var_mu_hat_normal = sigma2_hat / T
var_sigma2_hat_normal = (2 * sigma2_hat ** 2) / T

# Standard Errors
se_mu_hat_normal = np.sqrt(var_mu_hat_normal)
se_sigma2_hat_normal = np.sqrt(var_sigma2_hat_normal)

# Output results
print("\nUnder Normality Assumption:")
print(f"Estimated μ: {mu_hat:.6f}, Standard Error: {se_mu_hat_normal:.6f}, Variance: {var_mu_hat_normal:.6f}")
print(f"Estimated σ²: {sigma2_hat:.6f}, Standard Error: {se_sigma2_hat_normal:.6f}, Variance: {var_sigma2_hat_normal:.6f}")


# Save to file
results = pd.DataFrame({
    'Statistic': ['Mu Var (normal)', 'Var of Var (normal)', 'Standard Error of Mu (normal)', 'Standard Error of Variance (normal)'],
    'Value': [var_mu_hat_normal, var_sigma2_hat_normal, se_mu_hat_normal, se_sigma2_hat_normal]
})

# Save to file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results.to_excel(writer, sheet_name='1b)Normality_Var_SE', index=False)


In [ ]:
# 1c)
# Create a lagged series and remove 1st obs from original series
f_t_lag = f_t[:-1]
f_t = f_t[1:]

# Calculate the autocovariance matrix gamma
gamma = (f_t.T @ f_t_lag) / T

# Calculate new variance-covariance matrix S with the Newey-West estimator. One lag -> m = 1
S_NW = S_T + (1 - 1 / 2) * (gamma + gamma.T)

# Similarily to previously, D_T' @ D_T becomes identity matrix so we're left with
V_theta_hat_NW = S_NW / T

# Standard Errors
se_mu_hat_nw = np.sqrt(V_theta_hat_NW[0, 0])
se_sigma2_hat_nw = np.sqrt(V_theta_hat_NW[1, 1])

# Output results
print("\nNewey-West Standard Errors with One Lag:")
print(f"Estimated μ: {mu_hat:.6f}, Standard Error: {se_mu_hat_nw:.6f}")
print(f"Estimated σ²: {sigma2_hat:.6f}, Standard Error: {se_sigma2_hat_nw:.6f}")
print(f"Var-Covar Matrix - 0 0: {V_theta_hat_NW[0, 0]} and 0 1: {V_theta_hat_NW[0, 1]} and 1 0: {V_theta_hat_NW[1, 0]} and 1 1: {V_theta_hat_NW[1, 1]} ")


# Save to file
results = pd.DataFrame({
    'Statistic': ['Standard Error of Mu (normal)', 'Standard Error of Variance (normal)'],
    'Value': [se_mu_hat_nw, se_sigma2_hat_nw]
})

# Save to file
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results.to_excel(writer, sheet_name='1c)SE_NW', index=False)


# Question 2

In [ ]:
# 3a)
# The purpose of the J-test is to estimate if the  rest of the moment conditions (the ones that are overidentified) satisfy the null hypothesis. 
# In this case, the 3rd (skewness) and 4th (kurtosis) are the 2 excess moments, the distribution in this question is normal
# Which reminds me of the Jarque-Bera tests that uses the skewness and kurtosis to test for normality.

# 3b)
# Skewness and Kurtosis from moment conditions
sigma_hat = np.sqrt(sigma2_hat)

M3 = np.mean((returns - mu_hat) ** 3)
M4 = np.mean((returns - mu_hat) ** 4) - 3 * (sigma_hat ** 4)

# J test
g_T_over = np.array([M3, M4])

# S_T matrix for overidentified moments
f_t_over = np.column_stack([
    (returns - mu_hat) ** 3,   
    (returns - mu_hat) ** 4 - 3 * (sigma_hat ** 4)  
])

S_T_over = np.cov(f_t_over, rowvar=False, bias=True)

# J statistic
S_T_inv = np.linalg.inv(S_T_over)

J_T = T * (g_T_over @ S_T_inv @ g_T_over)

# JB Statistic
skewness = M3 / sigma_hat ** 3
kurtosis = M4 / sigma_hat ** 4 + 3
JB = (T / 6) * (skewness ** 2 + ((kurtosis - 3) ** 2) / 4)

# Chi2 test
alpha = 0.05
df = 2 # q = 4, k = 2
critical_value = stats.chi2.ppf(1 - alpha, df)

# Output results
print(f"Sample Skewness (S): {skewness:.6f}")
print(f"Sample Kurtosis (K): {kurtosis:.6f}")
print(f"Jarque-Bera Test Statistic (JB): {JB:.6f}")
print(f"J-Test Statistic (J_T): {J_T:.6f}")
print(f"Critical Value (Chi-squared, df=2, alpha=0.05): {critical_value:.6f}")


# Question 3

In [ ]:
# 3a)

# Simulate data
np.random.seed(1234)
n = 100
data = np.random.normal(loc = 0, scale = 1, size = n)

# Mean, stdev, and SE, 
mu_hat = data.mean()
sigma_hat = data.std()
se_mu = sigma_hat / np.sqrt(n)

# SE under true parameters
se_asymptotic = 1 / np.sqrt(n)

print(f"Sample Mean: {mu_hat:.6f}")
print(f"Sample Standard Deviation: {sigma_hat:.6f}")
print(f"Standard Error (Sample): {se_mu:.6f}")
print(f"Asymptotic Standard Error: {se_asymptotic:.6f}")

# Bootstrap
B = 50000
bootstrap_means = np.zeros(B)

for b in range(B):
    current_sample = np.random.choice(data, size = n, replace = True)
    bootstrap_means[b] = current_sample.mean()

# Bootstrap statistics
bootstrap_mean = bootstrap_means.mean()
bootstrap_se = bootstrap_means.std()
bootstrap_bias = bootstrap_mean - mu_hat
CI_lower = np.percentile(bootstrap_means, 2.5)
CI_upper = np.percentile(bootstrap_means, 97.5)

print("\nBootstrap Results:")
print(f"Bootstrap Mean: {bootstrap_mean:.6f}")
print(f"Bootstrap SE: {bootstrap_se:.6f}")
print(f"Bias: {bootstrap_bias:.6f}")
print(f"95% Confidence Interval: [{CI_lower:.6f}, {CI_upper:.6f}]")

In [ ]:
# 3b)

# Simulate AR(1) data   
np.random.seed(444)
n = 100
rho = 0.5

# Standard deviation of errors with unconditional variance of x as 1
unconditional_variance_x = 1
sigma_epsilon = np.sqrt(unconditional_variance_x*(1 - rho ** 2))

# Generate random errors with mean 0 and sigma_epsilon
epsilon = np.random.normal(loc = 0, scale = sigma_epsilon, size = n)

# Create random sample, autocorrelation of 0.5 + randomly generated error
x = np.zeros(n)
for t in range(1, n):
    x[t] = rho * x[t-1] + epsilon[t]

# Sample mean and SE
mu_hat = x.mean()
se_hat = x.std()
se_iid = se_hat / np.sqrt(n)

# Asymptotic SE for true values
se_asymptotic_true = np.sqrt(3 / n)

print(f"Sample Mean: {mu_hat:.6f}")
print(f"Standard Error (IID): {se_iid:.6f}")
print(f"Asymptotic SE (AR1): {se_asymptotic_true:.6f}")

# Bootstrap (Regular)
B = 50000
iid_bootstrap_means = np.zeros(B)

# Generate random samples from x
for b in range(B):
    current_sample = np.random.choice(x, size = n, replace = True)
    iid_bootstrap_means[b] = current_sample.mean()

iid_bootstrap_mean = iid_bootstrap_means.mean()
iid_bootstrap_se = iid_bootstrap_means.std()
bias = iid_bootstrap_mean - mu_hat
CI_lower = np.percentile(iid_bootstrap_means, 2.5)
CI_upper = np.percentile(iid_bootstrap_means, 97.5)

print("\nIID Bootstrap Results:")
print(f"Bootstrap Mean: {iid_bootstrap_mean:.6f}")
print(f"Bootstrap SE: {iid_bootstrap_se:.6f}")
print(f"Bias: {bias:.6f}")
print(f"95% Confidence Interval: [{CI_lower:.6f}, {CI_upper:.6f}]")


# Bootstrap (blocks)
block_size = 5
num_blocks = n - block_size + 1 # number of possible blocks
blocks = np.array([x[i:i + block_size] for i in range(num_blocks)]) # Create all possible blocks
required_blocks = int(np.ceil(n / block_size))  # Calculate required number of blocks
bootstrap_means_block = np.zeros(B)

for b in range(B):
    block_ind = np.random.choice(num_blocks, size = required_blocks, replace = True) # Generate 20 random numbers that represent selected blocks
    sample_blocks = blocks[block_ind] # Get the 20 random blocks with corresponding numbers from block_ind
    sample = np.concatenate(sample_blocks)[:n] 
    bootstrap_means_block[b] = sample.mean()

bootstrap_mean_block = bootstrap_means_block.mean()
bootstrap_mean_block_SE = bootstrap_means_block.std()
bias_block = bootstrap_mean_block - mu_hat
CI_lower_block = np.percentile(bootstrap_means_block, 2.5)
CI_upper_block = np.percentile(bootstrap_means_block, 97.5)

print("\nMoving Block Bootstrap Results:")
print(f"Bootstrap Mean: {bootstrap_mean_block:.6f}")
print(f"Bootstrap SE: {bootstrap_mean_block_SE:.6f}")
print(f"Bias: {bias_block:.6f}")
print(f"95% Confidence Interval: [{CI_lower_block:.6f}, {CI_upper_block:.6f}]")


